In [ ]:
# 导入必要的库
import h5py
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
from scipy.optimize import minimize

In [ ]:
# 加载数据集A（TRAIN38.mat的data）
def load_dataset_A(file_path):
    """加载TRAIN38.mat中的data作为数据集A"""
    print(f"加载数据集A: {file_path}")
    
    try:
        f = h5py.File(file_path, 'r')
        # 从文件中读取data矩阵
        data = np.array(f['data'])
        # 转置data（根据原代码的处理方式）
        data = data.transpose()
        f.close()
        
        print(f"数据集A加载完成:")
        print(f"  形状: {data.shape}")
        print(f"  数据类型: {data.dtype}")
        print(f"  统计信息: min={np.min(data)}, max={np.max(data)}, mean={np.mean(data)}, std={np.std(data)}")
        
        return data
    
    except Exception as e:
        print(f"加载数据集A时出错: {e}")
        return None

# 请修改为您的实际文件路径
file_path = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat'
if os.path.exists(file_path):
    dataset_A = load_dataset_A(file_path)
else:
    print(f"文件不存在: {file_path}")
    print("请设置正确的文件路径")

In [ ]:
# 加载数据集B（merged目录下的数据）
def load_dataset_B(base_dir):
    """加载merged目录下的所有npy文件作为数据集B"""
    print(f"加载数据集B: {base_dir}")
    
    merged_dir = os.path.join(base_dir, 'merged')
    
    # 检查目录是否存在
    if not os.path.exists(merged_dir):
        print(f"目录不存在: {merged_dir}")
        return None
    
    # 获取所有标签文件
    voxel_files = glob.glob(os.path.join(merged_dir, "label_*_count_*_voxels.npy"))
    
    if not voxel_files:
        print(f"在 {merged_dir} 中未找到体素文件")
        return None
    
    # 加载所有体素数据并合并
    all_data = []
    all_labels = []
    
    for voxel_file in tqdm(voxel_files, desc=f"加载merged数据"):
        # 从文件名提取标签ID
        filename = os.path.basename(voxel_file)
        parts = filename.split('_')
        label_id = int(parts[1])
        
        # 加载体素数据
        voxels = np.load(voxel_file)
        
        # 将标签ID扩展为与体素数据相同的行数
        labels = np.full((voxels.shape[0], 1), label_id)
        
        all_data.append(voxels)
        all_labels.append(labels)
    
    if all_data:
        combined_data = np.vstack(all_data)
        combined_labels = np.vstack(all_labels)
        
        print(f"数据集B加载完成:")
        print(f"  数据形状: {combined_data.shape}")
        print(f"  标签形状: {combined_labels.shape}")
        print(f"  数据类型: {combined_data.dtype}")
        print(f"  统计信息: min={np.min(combined_data)}, max={np.max(combined_data)}, mean={np.mean(combined_data)}, std={np.std(combined_data)}")
        
        return {
            'data': combined_data,
            'labels': combined_labels
        }
    else:
        print("未加载到任何数据")
        return None

# 请修改为您的实际目录路径
restructured_base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured'
if os.path.exists(restructured_base_dir):
    dataset_B_result = load_dataset_B(restructured_base_dir)
    if dataset_B_result is not None:
        dataset_B = dataset_B_result['data']
else:
    print(f"目录不存在: {restructured_base_dir}")
    print("请设置正确的目录路径")

In [ ]:
# 比较数据集A和数据集B的统计特性
def compare_datasets(A, B):
    """比较两个数据集的统计特性"""
    if A is None or B is None:
        print("无法比较数据集，请确保两个数据集已正确加载")
        return
    
    print("\n=== 数据集比较 ===")
    print(f"数据集A形状: {A.shape}")
    print(f"数据集B形状: {B.shape}")
    
    # 检查特征数量是否相同
    if A.shape[1] == B.shape[1]:
        print("两个数据集的特征数量相同")
        
        # 计算每个特征的统计信息
        A_means = np.mean(A, axis=0)
        A_stds = np.std(A, axis=0)
        B_means = np.mean(B, axis=0)
        B_stds = np.std(B, axis=0)
        
        # 显示一些统计信息比较
        print("\n前10个特征的统计信息比较:")
        for i in range(min(10, A.shape[1])):
            print(f"特征 {i+1}:")
            print(f"  数据集A: 均值={A_means[i]:.4f}, 标准差={A_stds[i]:.4f}")
            print(f"  数据集B: 均值={B_means[i]:.4f}, 标准差={B_stds[i]:.4f}")
            print(f"  差异: 均值差={B_means[i]-A_means[i]:.4f}, 标准差比={B_stds[i]/A_stds[i]:.4f}")
        
        # 计算整体统计量的差异
        mean_diff = np.mean(np.abs(B_means - A_means))
        std_ratio = np.mean(B_stds / (A_stds + 1e-10))  # 添加小值避免除零
        
        print(f"\n整体统计差异:")
        print(f"  均值绝对差异平均值: {mean_diff:.4f}")
        print(f"  标准差比率平均值: {std_ratio:.4f}")
        
        # 可视化一些特征的分布
        n_features_to_plot = min(3, A.shape[1])
        fig, axes = plt.subplots(n_features_to_plot, 2, figsize=(14, 4*n_features_to_plot))
        
        for i in range(n_features_to_plot):
            # 直方图
            axes[i, 0].hist(A[:, i], bins=50, alpha=0.5, label='数据集A')
            axes[i, 0].hist(B[:, i], bins=50, alpha=0.5, label='数据集B')
            axes[i, 0].set_title(f'特征 {i+1} 分布')
            axes[i, 0].legend()
            
            # 箱线图
            box_data = [A[:, i], B[:, i]]
            axes[i, 1].boxplot(box_data, labels=['数据集A', '数据集B'])
            axes[i, 1].set_title(f'特征 {i+1} 箱线图')
        
        plt.tight_layout()
        plt.show()
        
        return {
            'A_means': A_means,
            'A_stds': A_stds,
            'B_means': B_means,
            'B_stds': B_stds,
            'mean_diff': mean_diff,
            'std_ratio': std_ratio
        }
    else:
        print(f"警告: 两个数据集的特征数量不同! A: {A.shape[1]}, B: {B.shape[1]}")
        return None

# 如果已加载数据集A和数据集B，则比较它们
if 'dataset_A' in globals() and 'dataset_B' in globals():
    if dataset_A is not None and dataset_B is not None:
        comparison_results = compare_datasets(dataset_A, dataset_B)

In [ ]:
# 优化的StandardScaler实现
class OptimizedScaler:
    """优化的StandardScaler，可以调整mean和std参数以最小化两个数据集之间的MSE"""
    
    def __init__(self, initial_mean=None, initial_std=None):
        """
        初始化优化的StandardScaler
        
        参数:
            initial_mean: 初始均值参数，可选
            initial_std: 初始标准差参数，可选
        """
        self.mean_ = initial_mean
        self.scale_ = initial_std
        self.n_features_ = None
    
    def transform(self, X):
        """
        使用当前参数转换X
        
        参数:
            X: 要转换的数据集，形状为(n_samples, n_features)
        
        返回:
            X_scaled: 转换后的数据集
        """
        X = np.asarray(X)
        return (X - self.mean_) / self.scale_
    
    def fit_transform(self, X, target_X, **kwargs):
        """
        先fit再transform
        
        参数:
            X: 原始数据集
            target_X: 目标数据集
            **kwargs: 传递给fit方法的其他参数
        
        返回:
            X_scaled: 转换后的数据集
        """
        return self.fit(X, target_X, **kwargs).transform(X)
    
    def calculate_mse(self, X, target_X):
        """
        计算转换后的X与target_X之间的MSE
        
        参数:
            X: 原始数据集
            target_X: 目标数据集
            
        返回:
            mse: 均方误差
        """
        X_scaled = self.transform(X)
        return np.mean((X_scaled - target_X) ** 2)
    
    def fit_scipy(self, X, target_X, method='L-BFGS-B', max_iter=1000, verbose=True):
        """
        使用SciPy的优化器进行优化
        
        参数:
            X: 原始数据集，形状为(n_samples, n_features)
            target_X: 目标数据集，形状与X相同
            method: SciPy优化方法
            max_iter: 最大迭代次数
            verbose: 是否打印进度信息
        
        返回:
            self: 返回自身
        """
        # 检查输入
        X = np.asarray(X)
        target_X = np.asarray(target_X)
        
        if X.shape != target_X.shape:
            raise ValueError(f"X与target_X形状不同: {X.shape} vs {target_X.shape}")
        
        n_samples, n_features = X.shape
        self.n_features_ = n_features
        
        # 初始化参数
        if self.mean_ is None:
            self.mean_ = np.mean(X, axis=0)
        
        if self.scale_ is None:
            self.scale_ = np.std(X, axis=0)
            # 防止除以零
            self.scale_[self.scale_ == 0] = 1.0
        
        # 准备记录迭代过程
        self.loss_history_ = []
        
        # 定义目标函数
        def objective(params):
            mean_params = params[:n_features]
            std_params = params[n_features:]
            
            # 确保std为正
            std_params = np.maximum(std_params, 1e-10)
            
            # 临时设置参数
            self.mean_ = mean_params
            self.scale_ = std_params
            
            # 计算转换
            X_scaled = self.transform(X)
            
            # 计算MSE
            loss = np.mean((X_scaled - target_X) ** 2)
            self.loss_history_.append(loss)
            
            return loss
        
        # 优化
        initial_params = np.concatenate([self.mean_, self.scale_])
        
        if verbose:
            print("开始SciPy优化...")
            print(f"初始MSE: {self.calculate_mse(X, target_X):.6f}")
        
        result = minimize(
            objective, 
            initial_params, 
            method=method,
            options={'disp': verbose, 'maxiter': max_iter}
        )
        
        if verbose:
            print(f"优化完成: {result.message}")
            print(f"最终MSE: {result.fun:.6f}")
            print(f"迭代次数: {len(self.loss_history_)}")
        
        # 更新参数
        self.mean_ = result.x[:n_features]
        self.scale_ = np.maximum(result.x[n_features:], 1e-10)
        
        return self
    
    def fit_gradient_descent(self, X, target_X, max_iter=1000, tol=1e-6, learning_rate=0.01, verbose=True):
        """
        通过梯度下降优化mean和std参数，使转换后的X与target_X之间的MSE最小
        
        参数:
            X: 原始数据集，形状为(n_samples, n_features)
            target_X: 目标数据集，形状与X相同
            max_iter: 最大迭代次数
            tol: 容忍度，如果MSE变化小于此值，则停止迭代
            learning_rate: 学习率
            verbose: 是否打印进度信息
        
        返回:
            self: 返回自身
        """
        # 检查输入
        X = np.asarray(X)
        target_X = np.asarray(target_X)
        
        if X.shape != target_X.shape:
            raise ValueError(f"X与target_X形状不同: {X.shape} vs {target_X.shape}")
        
        n_samples, n_features = X.shape
        self.n_features_ = n_features
        
        # 初始化参数
        if self.mean_ is None:
            self.mean_ = np.mean(X, axis=0)
        
        if self.scale_ is None:
            self.scale_ = np.std(X, axis=0)
            # 防止除以零
            self.scale_[self.scale_ == 0] = 1.0
        
        # 准备记录迭代过程
        self.loss_history_ = []
        
        # 梯度下降优化
        prev_loss = float('inf')
        
        if verbose:
            print("开始梯度下降优化...")
            print(f"初始MSE: {self.calculate_mse(X, target_X):.6f}")
        
        for i in range(max_iter):
            # 计算当前转换
            X_scaled = (X - self.mean_) / self.scale_
            
            # 计算损失
            loss = np.mean((X_scaled - target_X) ** 2)
            self.loss_history_.append(loss)
            
            if verbose and (i % 10 == 0 or i == max_iter - 1):
                print(f"迭代 {i}, MSE: {loss:.6f}")
            
            # 检查收敛
            if abs(prev_loss - loss) < tol:
                if verbose:
                    print(f"收敛于迭代 {i}, MSE: {loss:.6f}")
                break
            
            prev_loss = loss
            
            # 计算梯度
            diff = X_scaled - target_X
            grad_mean = -2 * np.mean(diff / self.scale_.reshape(1, -1), axis=0)
            grad_scale = -2 * np.mean(diff * (-(X - self.mean_.reshape(1, -1)) / (self.scale_.reshape(1, -1) ** 2)), axis=0)
            
            # 更新参数
            self.mean_ -= learning_rate * grad_mean
            self.scale_ -= learning_rate * grad_scale
            
            # 确保std始终为正数
            self.scale_ = np.maximum(self.scale_, 1e-10)
        
        if verbose:
            print(f"最终MSE: {loss:.6f}")
            print(f"迭代次数: {len(self.loss_history_)}")
        
        return self
    
    def fit(self, X, target_X, algorithm='scipy', **kwargs):
        """
        通过选择的算法优化mean和std参数
        
        参数:
            X: 原始数据集
            target_X: 目标数据集
            algorithm: 优化算法，'scipy'或'gradient_descent'
            **kwargs: 传递给具体优化方法的其他参数
        
        返回:
            self: 返回自身
        """
        if algorithm.lower() == 'scipy':
            return self.fit_scipy(X, target_X, **kwargs)
        elif algorithm.lower() == 'gradient_descent':
            return self.fit_gradient_descent(X, target_X, **kwargs)
        else:
            raise ValueError(f"不支持的算法: {algorithm}，请使用'scipy'或'gradient_descent'")
    
    def save(self, file_path):
        """保存优化后的scaler到文件"""
        if not hasattr(self, 'mean_') or not hasattr(self, 'scale_'):
            print("无法保存scaler，请先进行fit")
            return False
        
        # 准备要保存的数据
        data = {
            'mean_': self.mean_,
            'scale_': self.scale_,
            'n_features_': self.n_features_
        }
        
        if hasattr(self, 'loss_history_'):
            data['loss_history_'] = self.loss_history_
        
        try:
            np.savez(file_path, **data)
            print(f"优化后的scaler已保存到: {file_path}")
            return True
        except Exception as e:
            print(f"保存scaler时出错: {e}")
            return False
    
    @classmethod
    def load(cls, file_path):
        """从文件加载优化后的scaler"""
        try:
            data = np.load(file_path)
            scaler = cls()
            scaler.mean_ = data['mean_']
            scaler.scale_ = data['scale_']
            scaler.n_features_ = data['n_features_']
            
            if 'loss_history_' in data:
                scaler.loss_history_ = data['loss_history_']
            
            print(f"优化后的scaler已从 {file_path} 加载")
            return scaler
        except Exception as e:
            print(f"加载scaler时出错: {e}")
            return None

In [ ]:
# 应用优化的scaler
def apply_optimized_scaler(A, B):
    """应用优化的scaler将数据集A转换为接近数据集B"""
    if A is None or B is None:
        print("无法应用优化的scaler，请确保两个数据集已正确加载")
        return
    
    print("\n=== 应用优化的Scaler ===")
    
    # 检查特征数量是否相同
    if A.shape[1] == B.shape[1]:
        # 使用标准的StandardScaler作为基准
        print("\n使用标准StandardScaler:")
        standard_scaler = StandardScaler()
        standard_scaler.fit(A)
        A_std = standard_scaler.transform(A)
        
        # 计算MSE
        mse_standard = mean_squared_error(A_std, B)
        print(f"标准StandardScaler的MSE: {mse_standard:.6f}")
        
        # 使用随机抽样减少数据量以加速优化
        sample_size = min(100000, A.shape[0], B.shape[0])
        print(f"\n使用{sample_size}个样本进行优化:")
        indices_A = np.random.choice(A.shape[0], sample_size, replace=False)
        indices_B = np.random.choice(B.shape[0], sample_size, replace=False)
        
        A_sample = A[indices_A]
        B_sample = B[indices_B]
        
        # 使用优化的Scaler - 梯度下降方法
        print("\n使用优化的Scaler (梯度下降):")
        optimized_scaler_gd = OptimizedScaler()
        optimized_scaler_gd.fit(A_sample, B_sample, algorithm='gradient_descent', max_iter=500, learning_rate=0.001, verbose=True)
        
        # 在整个数据集上计算MSE
        A_opt_gd = optimized_scaler_gd.transform(A)
        mse_optimized_gd = mean_squared_error(A_opt_gd, B)
        print(f"优化的Scaler (梯度下降) 在全数据集上的MSE: {mse_optimized_gd:.6f}")
        print(f"相对标准Scaler的改进: {(1 - mse_optimized_gd/mse_standard) * 100:.2f}%")
        
        # 使用优化的Scaler - SciPy优化方法
        print("\n使用优化的Scaler (SciPy优化):")
        optimized_scaler_scipy = OptimizedScaler()
        optimized_scaler_scipy.fit(A_sample, B_sample, algorithm='scipy', verbose=True)
        
        # 在整个数据集上计算MSE
        A_opt_scipy = optimized_scaler_scipy.transform(A)
        mse_optimized_scipy = mean_squared_error(A_opt_scipy, B)
        print(f"优化的Scaler (SciPy优化) 在全数据集上的MSE: {mse_optimized_scipy:.6f}")
        print(f"相对标准Scaler的改进: {(1 - mse_optimized_scipy/mse_standard) * 100:.2f}%")
        
        # 可视化比较结果
        print("\n可视化比较:")
        
        # 绘制损失曲线
        plt.figure(figsize=(10, 6))
        plt.plot(optimized_scaler_gd.loss_history_, label='梯度下降损失')
        plt.plot(optimized_scaler_scipy.loss_history_, label='SciPy优化损失')
        plt.axhline(y=mse_standard, color='r', linestyle='-', label='标准Scaler MSE')
        plt.xlabel('迭代次数')
        plt.ylabel('MSE')
        plt.title('优化过程中的MSE变化')
        plt.legend()
        plt.grid(True)
        plt.show()
        
        # 比较分布（随机抽样一小部分数据）
        vis_sample_size = min(10000, A.shape[0], B.shape[0])
        vis_indices_A = np.random.choice(A.shape[0], vis_sample_size, replace=False)
        vis_indices_B = np.random.choice(B.shape[0], vis_sample_size, replace=False)
        
        A_vis = A[vis_indices_A]
        B_vis = B[vis_indices_B]
        A_std_vis = A_std[vis_indices_A]
        A_opt_gd_vis = A_opt_gd[vis_indices_A]
        A_opt_scipy_vis = A_opt_scipy[vis_indices_A]
        
        n_features_to_plot = min(3, A.shape[1])
        fig, axes = plt.subplots(n_features_to_plot, 1, figsize=(12, 4*n_features_to_plot))
        
        for i in range(n_features_to_plot):
            feature_idx = i
            
            if n_features_to_plot == 1:
                ax = axes
            else:
                ax = axes[i]
            
            ax.hist(A_vis[:, feature_idx], bins=50, alpha=0.3, label='原始数据集A')
            ax.hist(A_std_vis[:, feature_idx], bins=50, alpha=0.3, label='标准Scaler')
            ax.hist(A_opt_gd_vis[:, feature_idx], bins=50, alpha=0.3, label='优化Scaler (GD)')
            ax.hist(A_opt_scipy_vis[:, feature_idx], bins=50, alpha=0.3, label='优化Scaler (SciPy)')
            ax.hist(B_vis[:, feature_idx], bins=50, alpha=0.3, label='目标数据集B')
            ax.set_title(f'特征 {feature_idx+1} 分布比较')
            ax.legend()
        
        plt.tight_layout()
        plt.show()
        
        # 返回最佳的scaler
        if mse_optimized_gd < mse_optimized_scipy:
            print("\n梯度下降方法表现更好")
            best_scaler = optimized_scaler_gd
        else:
            print("\nSciPy优化方法表现更好")
            best_scaler = optimized_scaler_scipy
        
        # 保存最佳scaler
        best_scaler.save('optimized_scaler.npz')
        
        return best_scaler
    else:
        print(f"警告: 两个数据集的特征数量不同! A: {A.shape[1]}, B: {B.shape[1]}")
        return None

# 如果已加载数据集A和数据集B，则应用优化的scaler
if 'dataset_A' in globals() and 'dataset_B' in globals():
    if dataset_A is not None and dataset_B is not None:
        best_scaler = apply_optimized_scaler(dataset_A, dataset_B)

In [ ]:
# 使用优化的scaler示例
def scaler_usage_example(best_scaler=None):
    """展示如何使用优化的scaler进行数据转换和评估"""
    print("\n=== 优化的Scaler使用示例 ===")
    
    # 如果没有提供scaler，则尝试加载
    if best_scaler is None:
        if os.path.exists('optimized_scaler.npz'):
            best_scaler = OptimizedScaler.load('optimized_scaler.npz')
        else:
            print("未找到保存的scaler文件，请先运行优化")
            return
    
    # 生成一些测试数据
    np.random.seed(42)
    test_data = np.random.randn(1000, best_scaler.n_features_)
    
    # 使用优化的scaler转换数据
    transformed_data = best_scaler.transform(test_data)
    
    # 显示转换前后的统计信息
    print("\n转换前的统计信息:")
    print(f"  均值: {np.mean(test_data):.4f}")
    print(f"  标准差: {np.std(test_data):.4f}")
    
    print("\n转换后的统计信息:")
    print(f"  均值: {np.mean(transformed_data):.4f}")
    print(f"  标准差: {np.std(transformed_data):.4f}")
    
    # 可视化转换前后的分布（随机选择几个特征）
    n_features_to_plot = min(3, best_scaler.n_features_)
    feature_indices = np.random.choice(best_scaler.n_features_, n_features_to_plot, replace=False)
    
    fig, axes = plt.subplots(n_features_to_plot, 1, figsize=(10, 4*n_features_to_plot))
    
    for i, feature_idx in enumerate(feature_indices):
        if n_features_to_plot == 1:
            ax = axes
        else:
            ax = axes[i]
        
        ax.hist(test_data[:, feature_idx], bins=30, alpha=0.5, label='原始数据')
        ax.hist(transformed_data[:, feature_idx], bins=30, alpha=0.5, label='转换后数据')
        ax.set_title(f'特征 {feature_idx+1} 转换前后分布比较')
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    print("\n如何在实际应用中使用优化的scaler:")
    print("""
    # 1. 加载保存的scaler
    optimized_scaler = OptimizedScaler.load('optimized_scaler.npz')
    
    # 2. 使用scaler转换新数据
    transformed_data = optimized_scaler.transform(new_data)
    
    # 3. 后续处理...
    """)

# 如果已获取最佳scaler，则运行使用示例
if 'best_scaler' in globals() and best_scaler is not None:
    scaler_usage_example(best_scaler)
else:
    # 尝试加载保存的scaler
    if os.path.exists('optimized_scaler.npz'):
        loaded_scaler = OptimizedScaler.load('optimized_scaler.npz')
        scaler_usage_example(loaded_scaler)